<a href="https://colab.research.google.com/github/TViguini/agentes-2-equipe-01/blob/main/enc02_Thiago.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q "openai>=1.99.0,<3"

import importlib.metadata as md
print("openai", md.version("openai"))

openai 2.45.0


In [3]:
import os

def obter_chave(nome: str) -> str:
    """Le um segredo dos Secrets do Colab; fora do Colab, da variavel de ambiente."""
    try:
        from google.colab import userdata
        return userdata.get(nome)
    except ImportError:
        valor = os.getenv(nome)
        if not valor:
            raise RuntimeError(f"Defina {nome} nos Secrets do Colab ou no ambiente.")
        return valor

GROQ_KEY = obter_chave("GROQ_API_KEY")

# Confirma que a chave chegou, sem imprimi-la. Nunca imprima uma chave inteira.
print("chave carregada, termina em:", GROQ_KEY[-4:])

chave carregada, termina em: Qe6G


In [4]:
import json

# Nada nesta celula vai para o modelo. Aqui so ensaiamos as quatro construcoes
# que voce vai usar o semestre todo. A primeira chamada de verdade e na Parte 3.

def temperatura_reator(reator: str) -> str:
    """Le a temperatura atual de um reator da planta.

    Args:
        reator: identificador do reator, por exemplo R-101
    """
    leituras = {"R-101": 87.4, "R-102": 91.2, "R-103": 78.9}
    if reator not in leituras:
        return f"reator {reator} desconhecido"
    return f"{reator}: {leituras[reator]} graus Celsius"


# 1) A funcao anotada, chamada por voce, do jeito normal.
print(temperatura_reator("R-101"))
print(temperatura_reator("R-999"))

# 2) A docstring nao e comentario: e dado, legivel em tempo de execucao.
#    No Encontro 5 e este texto que viaja junto para o modelo, e e por ele
#    que o modelo decide se esta ferramenta serve para a pergunta que recebeu.
print("\n--- o que o modelo vai ler sobre esta ferramenta ---")
print(temperatura_reator.__doc__)

# 3) Dicionario e JSON: o corpo exato da requisicao que a Parte 3 vai enviar.
requisicao = {
    "model": "llama-3.1-8b-instant",
    "messages": [
        {"role": "system", "content": "Você responde a engenheiros. Seja preciso e breve."},
        {"role": "user", "content": "Qual a temperatura do reator R-101?"},
    ],
}
print("\n--- o que sai da sua maquina pela rede ---")
print(json.dumps(requisicao, ensure_ascii=False, indent=2))

R-101: 87.4 graus Celsius
reator R-999 desconhecido

--- o que o modelo vai ler sobre esta ferramenta ---
Le a temperatura atual de um reator da planta.

    Args:
        reator: identificador do reator, por exemplo R-101
    

--- o que sai da sua maquina pela rede ---
{
  "model": "llama-3.1-8b-instant",
  "messages": [
    {
      "role": "system",
      "content": "Você responde a engenheiros. Seja preciso e breve."
    },
    {
      "role": "user",
      "content": "Qual a temperatura do reator R-101?"
    }
  ]
}


In [5]:
from openai import OpenAI

# --- as tres linhas que definem o provedor ---
LLM_BASE_URL = "https://api.groq.com/openai/v1"
LLM_MODEL = "llama-3.1-8b-instant"
LLM_API_KEY = GROQ_KEY
# ---------------------------------------------

cliente = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)

resposta = cliente.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "system", "content": "Você responde a engenheiros. Seja preciso e breve."},
        {"role": "user", "content": "Em uma frase: o que diferencia um agente de um chatbot?"},
    ],
)

print(resposta.choices[0].message.content)

Um agente é um sistema computacional capaz de interagir de forma significativa com o ambiente, enquanto um chatbot é um aplicativo que utiliza inteligência artificial para simular conversas com usuários, mas não necessariamente possui agência ou capacidade de interagir com o ambiente externo.


In [6]:
modelos = sorted(m.id for m in cliente.models.list())
print(f"{len(modelos)} modelos disponiveis neste provedor:\n")
for m in modelos:
    print("  ", m)

15 modelos disponiveis neste provedor:

   allam-2-7b
   canopylabs/orpheus-arabic-saudi
   canopylabs/orpheus-v1-english
   groq/compound
   groq/compound-mini
   llama-3.1-8b-instant
   llama-3.3-70b-versatile
   meta-llama/llama-prompt-guard-2-22m
   meta-llama/llama-prompt-guard-2-86m
   openai/gpt-oss-120b
   openai/gpt-oss-20b
   openai/gpt-oss-safeguard-20b
   qwen/qwen3.6-27b
   whisper-large-v3
   whisper-large-v3-turbo


In [7]:
u = resposta.usage
print(f"entrada : {u.prompt_tokens} tokens")
print(f"saida   : {u.completion_tokens} tokens")
print(f"total   : {u.total_tokens} tokens")

TOKENS_PERGUNTA_SIMPLES = u.total_tokens
print("\nGuarde este numero para comparar no Encontro 14.")

entrada : 66 tokens
saida   : 66 tokens
total   : 132 tokens

Guarde este numero para comparar no Encontro 14.


In [8]:
def perguntar(pergunta: str, instrucao: str = "Você responde a engenheiros. Seja breve."):
    """Envia uma pergunta ao modelo e devolve (texto_da_resposta, total_de_tokens)."""
    r = cliente.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": instrucao},
            {"role": "user", "content": pergunta}
            # SEU CÓDIGO: acrescente a mensagem do usuario, com a pergunta recebida
        ],
    )
    texto = None    # SEU CÓDIGO: extraia o texto da resposta
    tokens = None   # SEU CÓDIGO: extraia o total de tokens

    texto = r.choices[0].message.content
    tokens = r.usage.total_tokens

    return texto, tokens


# Teste: as tres perguntas devem responder, e nenhuma deve imprimir None
for p in ("O que é PEAS?",
          "Cite um risco de usar agentes em malha de controle em tempo real.",
          "O modelo executa a ferramenta, ou apenas pede que ela seja executada?"):
    texto, tokens = perguntar(p)
    print(f"[{tokens} tokens] {p}\n  -> {texto}\n")

[174 tokens] O que é PEAS?
  -> PEAS é um acrônimo usado em engenharia de controle para descrever os componentes fundamentais de um controleador:

- P: Processo (ou processo que controlamos)
- E: Sensor de erro (calcula a diferença entre o valor desejado e o valor real do processo)
- A: Algoritmo ou controleador (processa o sinal de erro e gera um sinal de controle)
- S: Interface (ou atuador) que atua sobre o processo para mantê-lo no valor desejado.

[221 tokens] Cite um risco de usar agentes em malha de controle em tempo real.
  -> Um risco de usar agentes em malha de controle em tempo real é a possibilidade de "escritos indesejados" que possam levar ao "ouro nublado" (ouro nublado se refere à situação em que uma solução aparentemente otimizada não atinge seu objetivo original, causando efeitos inesperados).

Os agentes podem interagir de maneira inadequada, gerando comportamentos complexos e difíceis de entender, o que pode comprometer a estabilidade e a confiabilidade da malha de 

In [9]:
PERGUNTA = "Dê um nome curto para um agente que diagnostica falhas em uma planta industrial."

for temp in (0.0, 0.0, 1.2, 1.2):
    r = cliente.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": PERGUNTA}],
        temperature=temp,
        max_tokens=100,
    )
    print(f"temperatura {temp}: {r.choices[0].message.content.strip()}\n")

print("\nAs duas primeiras linhas tendem a coincidir; as duas ultimas, nao.")
print("Guarde a pergunta: como se testa um programa que nao repete a si mesmo?")

temperatura 0.0: Um nome curto para um agente que diagnostica falhas em uma planta industrial pode ser:

- "FaultFinder" (encontrador de falhas)
- "Diagno" (diagnóstico)
- "FaultScan" (escaneamento de falhas)
- "PlantaGuard" (guardião da planta)
- "FaultFix" (correção de falhas)

Escolha o que melhor se adequa ao seu contexto.

temperatura 0.0: Um nome curto para um agente que diagnostica falhas em uma planta industrial pode ser:

- "FaultFinder" (encontrador de falhas)
- "Diagno" (diagnóstico)
- "FaultScan" (escaneamento de falhas)
- "PlantaGuard" (guardião da planta)
- "FaultFix" (correção de falhas)

Escolha o que melhor se adequa ao seu contexto.

temperatura 1.2: Pode chamar um agente como isso de "Diagsys"

temperatura 1.2: Poderia chamar esse agente de "FaultDetect" ou "Inspectus"


As duas primeiras linhas tendem a coincidir; as duas ultimas, nao.
Guarde a pergunta: como se testa um programa que nao repete a si mesmo?


In [10]:
import time

CONFIGURACOES = {
    "pequeno": dict(base_url="https://api.groq.com/openai/v1",
                    model="llama-3.1-8b-instant", key=GROQ_KEY),
    "grande":  dict(base_url="https://api.groq.com/openai/v1",
                    model="llama-3.3-70b-versatile", key=GROQ_KEY),
}

PERGUNTA = ("Um operador relata vibração anormal na bomba P-204. "
            "Liste no máximo três hipóteses de causa, em ordem de probabilidade.")

for nome, cfg in CONFIGURACOES.items():
    try:
        c = OpenAI(base_url=cfg["base_url"], api_key=cfg["key"])
        t0 = time.time()
        r = c.chat.completions.create(
            model=cfg["model"],
            messages=[{"role": "user", "content": PERGUNTA}],
            temperature=0.0,
        )
        dt = time.time() - t0
        print(f"=== {nome}: {cfg['model']} | {r.usage.total_tokens} tokens | {dt:.1f}s ===")
        print(r.choices[0].message.content.strip(), "\n")
    except Exception as e:
        print(f"=== {nome} FALHOU: {type(e).__name__}: {str(e)[:160]}\n")

print("O modelo maior costuma organizar melhor, e custa mais tokens e mais tempo.")

=== pequeno: llama-3.1-8b-instant | 363 tokens | 0.6s ===
Aqui estão três hipóteses de causa para a vibração anormal na bomba P-204, em ordem de probabilidade:

1. **Desalinhamento ou deslocamento do eixo da bomba**: Isso pode ocorrer devido a vibrações externas, como movimentos de equipamentos próximos ou mudanças no nível de fluido. É uma hipótese provável, pois é uma causa comum de vibrações anormais em bombas.

2. **Desgaste ou corrosão de componentes**: O desgaste ou corrosão de componentes, como a rota de entrada ou saída da bomba, pode causar vibrações anormais. Isso pode ocorrer devido à exposição a fluidos corrosivos ou ao desgaste normal com o tempo.

3. **Problemas de balanceamento ou equilíbrio da bomba**: Se a bomba não estiver balanceada corretamente, pode causar vibrações anormais. Isso pode ocorrer devido a problemas de fabricação ou ajustes incorretos durante a instalação.

É importante notar que essas hipóteses devem ser investigadas e verificadas por um técnico quali

In [11]:
try:
    OpenAI(base_url=LLM_BASE_URL, api_key="chave-invalida-de-proposito").chat.completions.create(
        model=LLM_MODEL, messages=[{"role": "user", "content": "oi"}]
    )
except Exception as e:
    print("tipo:", type(e).__name__)
    print("mensagem:", str(e)[:220])
    print("\nEra o esperado. Reconhecer a mensagem economiza tempo depois.")

tipo: AuthenticationError
mensagem: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

Era o esperado. Reconhecer a mensagem economiza tempo depois.
